In [1]:
print(123)

123


In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [4]:
def llm(prompt):
    response = openai_client.responses.create(
        model="openai/gpt-oss-120b",
        input=prompt
    )
    return response.output_text

In [5]:
llm("Hey, what's up?")


'Hey there! I’m doing great—ready to help with any questions or topics you’ve got in mind. What’s up with you?'

In [6]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Absolutely—you can join the course at any time! Here’s what you need to do to get started:

1. **Create an Account / Log In**  
   - If you don’t already have one, sign up on our platform using your email or a social login.  
   - If you already have an account, just log in.

2. **Check the Prerequisites**  
   - Most of our courses are open‑access, but a few have recommended background knowledge (e.g., basic programming, introductory statistics).  
   - If the course you’re interested in lists any prerequisites, a quick review will help you get the most out of the material.

3. **Enroll in the Course**  
   - Navigate to the course page and click **“Enroll”** or **“Join Now.”**  
   - If the course is free, you’ll be added instantly. If it’s a paid track, you’ll be prompted to complete the payment process.

4. **Set Up Your Learning Environment**  
   - Make sure you have the required software or tools installed (often listed in the syllabus).  
   - Some courses include downloadable 

In [7]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [8]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [9]:
answer = llm(prompt)
print(answer)

Yes—you can still join the course. If you’d like to earn a certificate, just be sure to submit your project before the submission deadline.


In [10]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [11]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

print(courses_raw)

[{'course': 'machine-learning-zoomcamp', 'course_name': 'ML Zoomcamp', 'path': '/json/machine-learning-zoomcamp.json', 'questions_count': 472}, {'course': 'llm-zoomcamp', 'course_name': 'LLM Zoomcamp', 'path': '/json/llm-zoomcamp.json', 'questions_count': 79}, {'course': 'data-engineering-zoomcamp', 'course_name': 'Data Engineering Zoomcamp', 'path': '/json/data-engineering-zoomcamp.json', 'questions_count': 402}, {'course': 'mlops-zoomcamp', 'course_name': 'MLOps Zoomcamp', 'path': '/json/mlops-zoomcamp.json', 'questions_count': 255}]


In [12]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1208

In [13]:
documents[500]

{'id': '4d8c1f5b3a',
 'course': 'llm-zoomcamp',
 'section': 'Module 1: Introduction to LLMs and RAG',
 'question': 'How do I count tokens for a non-OpenAI model (Gemini, Mistral, HuggingFace)?',
 'answer': '`tiktoken` only ships tokenizers for OpenAI models. Using `cl100k_base` for other providers gives wrong counts and unreliable cost estimates.\n\nFor other providers, use their native tokenizer:\n\n- Gemini:\n  ```python\n  import google.generativeai as genai\n  model = genai.GenerativeModel(\'gemini-2.0-flash\')\n  print(model.count_tokens(prompt))\n  ```\n- Hugging Face / open-source models:\n  ```python\n  from transformers import AutoTokenizer\n  tok = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B")\n  print(len(tok.encode(prompt)))\n  ```\n- Mistral: use the official `mistral-common` tokenizer package.\n\nDon\'t use `cl100k_base` as a generic fallback — token counts will diverge from what the provider actually bills.'}

In [14]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [15]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [16]:
[doc["question"] for doc in search_results]


['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'When will the course be offered next?',
 'I missed the first homework - can I still get a certificate?']

In [17]:
results = index.search(
    question,
    num_results=5,
    boost_dict={"question": 2.0, "section": 0.5}
)

In [18]:
results = index.search(
    question,
    num_results=5,
    filter_dict={"course": "mlops-zoomcamp"}
)

In [19]:
[doc["question"] for doc in results]


['Course - Can I still join the course after the start date?',
 'Homework: Just found this course, can I still submit homeworks?',
 'I forgot if I registered, can I still join the zoomcamp?',
 'Certificate - Can I follow the course in a self-paced mode and get a certificate?',
 'Course: How do I start?']

In [20]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [21]:
search_results = search(question)


In [22]:
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [24]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [25]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [26]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [27]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [28]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [29]:
response = openai_client.responses.create(
    model="openai/gpt-oss-120b",
    input=prompt
)

In [33]:
response.output[0]


ResponseReasoningItem(id='resp_01ktey80t4ewhvhh1znq1pkh02', summary=[], type='reasoning', content=[Content(text='We need to answer: "I just discovered the course. Can I join now?" Based on context: answer: Yes, but need to submit project before submissions close to get certificate. Provide details.', type='reasoning_text')], encrypted_content=None, status='completed')

In [34]:
response.output[0].content[0].text


'We need to answer: "I just discovered the course. Can I join now?" Based on context: answer: Yes, but need to submit project before submissions close to get certificate. Provide details.'

In [35]:
response.output_text


'**Yes – you can still join the course.**  \n\n- **Start learning right away:** There’s no gate‑keeping registration list; you can begin the lessons and work on the homework as soon as you like.  \n- **Certificate eligibility:** If you want a certificate, you’ll need to **submit your final project before the submission deadline** (the deadline is the date when we stop accepting projects for the current cohort).  \n- **Peer‑review requirement:** After you submit, you must also complete **three peer‑reviews** of other students’ capstone projects. This can only be done while the cohort is active, so make sure you finish both the submission and the reviews before the cohort closes.  \n\nSo feel free to jump in now, work through the material, and plan to turn in your project before the cut‑off if you need the certification. Happy learning!'

In [36]:
response.usage


ResponseUsage(input_tokens=399, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=228, output_tokens_details=OutputTokensDetails(reasoning_tokens=41), total_tokens=627)

In [37]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.00132525

In [39]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = openai_client.responses.create(
    model="openai/gpt-oss-120b",
    input=message_history
)

In [43]:
def llm(instructions, user_prompt, model="openai/gpt-oss-120b"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [45]:
def rag(query, model="openai/gpt-oss-120b"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [46]:
rag(question)

'Yes—you can still join the course. Just keep in mind that if you want to earn a certificate, you’ll need to submit your project before the submission deadline.'